# 🎙️ 播客摘要器 — 第 3 周任务

**作者：** victorConqueror（维克多）

## 练习目标（理念）

把本地/云盘上的播客 **MP3** 变成可读摘要：

1. 用 HuggingFace **Whisper**（`pipeline`）做语音转文字（ASR）
2. 用 **Llama 3.2 3B Instruct**（4-bit 量化）对转录稿做结构化 Markdown 摘要
3. **全程本地/Hub 开源模型**，不调用 OpenAI API

## 和本课 Week 3 的关系

| 本课概念 | 本笔记里你会看到 |
|----------|------------------|
| HuggingFace `pipeline()` | `automatic-speech-recognition` 一键转录 |
| Tokenizer + Causal LM | `AutoTokenizer` / `AutoModelForCausalLM` |
| 4-bit 量化 | `BitsAndBytesConfig`（塞进免费 Colab T4） |
| 聊天模板 | `apply_chat_template()` |
| 流式生成 | `TextStreamer` 边生成边打印 |

## 怎么跑

1. 建议在 **Google Colab + T4 GPU** 上从上到下运行
2. HuggingFace 账号需申请 `meta-llama/Llama-3.2-3B-Instruct` 访问权，并把 `HF_TOKEN` 放进 Colab Secrets
3. 在 Drive 的 `llms/` 文件夹放好 MP3，并改下面的 `audio_filename`


## 步骤 0：安装依赖

需要特定版本的：

- `transformers==4.57.6`：加载 Whisper / Llama
- `bitsandbytes`：4-bit 量化
- `accelerate`：`device_map="auto"` 智能把层放到 GPU/CPU


In [ ]:
# ========== 安装：升级量化与加速库，并钉死 transformers 版本 ==========
# Colab 魔法命令：!pip —— 只影响环境，不改业务逻辑；版本号字符串保持原样
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6


## 步骤 1：导入

HuggingFace 关键类对照：

| 符号 | 是什么 | 为什么用 |
|------|--------|----------|
| `AutoTokenizer` | 文本 ↔ token id | 模型只吃数字序列 |
| `AutoModelForCausalLM` | 因果语言模型（生成下一 token） | Llama 摘要 |
| `TextStreamer` | 生成时实时打印 token | 像 ChatGPT 一样流式看输出 |
| `BitsAndBytesConfig` | 4-bit 量化配置 | 大模型塞进小 GPU |
| `pipeline` | 高级封装：模型+预处理+后处理 | Whisper 一行转录 |


In [ ]:
# ========== 导入：标准库 / PyTorch / Colab / HuggingFace ==========

# 标准库 os：查文件是否存在、算文件大小
import os
# PyTorch：张量、dtype（如 float16 / bfloat16）、设备
import torch
# 笔记本里漂亮渲染 Markdown
from IPython.display import Markdown, display
# Colab：挂载 Drive；userdata 读 Secrets（如 HF_TOKEN）
from google.colab import drive, userdata
# HuggingFace Hub 登录（门控模型下载需要）
from huggingface_hub import login
# transformers：分词器、因果 LM、流式打印、量化配置、pipeline 封装
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
    pipeline,
)


## 步骤 2：常量与模型选择

集中定义模型 id（字符串必须和 Hub 上的仓库名一致）：

- **Whisper**（`openai/whisper-medium.en`）：英文语音转文字
- **Llama 3.2**（`meta-llama/Llama-3.2-3B-Instruct`）：指令微调小模型，适合摘要


In [ ]:
# ========== 常量：模型仓库名集中写在一处，后面只改这里 ==========

# Whisper：HuggingFace Hub 上的语音识别模型 id（.en = 英文化版本）
WHISPER_MODEL = "openai/whisper-medium.en"
# Llama 3.2 3B Instruct：门控模型，需先申请访问；字符串勿改
LLAMA_MODEL = "meta-llama/Llama-3.2-3B-Instruct"


## 步骤 3：连接 Google Drive 并指向音频

挂载 Drive 后，笔记本才能读到你的 MP3。

**操作：**

1. 在 Google Drive 建文件夹 `llms`
2. 把播客 MP3 上传进去
3. 把下面代码里的 `audio_filename` 改成你的文件名


In [ ]:
# ========== 挂载 Drive，定位 MP3，并做存在性检查 ==========

# 挂载 Google Drive 到 /content/drive（Colab 会弹出授权）
drive.mount("/content/drive")

# 指向播客文件——改文件名以匹配你的 MP3；路径字符串影响能否读到文件
audio_filename = "/content/drive/MyDrive/llms/podcast_extract.mp3"

# 验证文件是否存在：存在则打印大小，否则给出上传提示（英文 print 保持原样）
if os.path.exists(audio_filename):
    # 字节 → MB，方便判断会不会转录太久
    file_size_mb = os.path.getsize(audio_filename) / (1024 * 1024)
    print(f"✅ Found audio file: {audio_filename}")
    print(f"📁 File size: {file_size_mb:.1f} MB")
else:
    print(f"❌ File not found: {audio_filename}")
    print("Please upload your podcast MP3 to Google Drive in the 'llms' folder.")


## 步骤 4：登录 HuggingFace Hub

Llama 3.2 是**门控模型（gated model）**，你需要：

1. 打开 [meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) 申请访问
2. 在 [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 创建 token
3. 在 Colab Secrets（左侧 🔑）添加名为 `HF_TOKEN` 的密钥


In [ ]:
# ========== 用 Colab Secrets 里的 HF_TOKEN 登录 Hub ==========

# 从 Colab userdata 读取密钥名 HF_TOKEN（不要把 token 写进笔记本正文）
hf_token = userdata.get('HF_TOKEN')
# login：让后续 from_pretrained / pipeline 能下载门控权重；add_to_git_credential 按原参数保留
login(hf_token, add_to_git_credential=True)
print("✅ Logged in to HuggingFace Hub!")


---

## 🎧 第 1 部分：用 Whisper 转录

### `pipeline()` 是什么？

把它当成**一行快捷方式**：

1. 从 Hub 下载模型
2. 下载匹配的处理器/特征提取器
3. 做音频预处理（波形 → 特征）
4. 跑推理
5. 把输出后处理成文本

没有 `pipeline()`，你通常要手写十几二十行样板代码。


In [ ]:
# ========== 创建 Whisper ASR pipeline（GPU + 半精度 + 时间戳） ==========

# task：automatic-speech-recognition = 语音识别
# model：用哪个 Whisper；dtype=float16 比 float32 更省显存
# device='cuda'：放到 GPU；return_timestamps=True：输出带时间信息
whisper_pipe = pipeline(
    task="automatic-speech-recognition",
    model=WHISPER_MODEL,
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True,
)

print("✅ Whisper pipeline loaded!")


In [ ]:
# ========== 跑转录：把文件路径交给 pipeline，取出 text ==========

# 提示开始转录（可能要一两分钟，取决于音频长度）
print("🎙️ Transcribing podcast... (this may take a minute)\n")

# 直接传路径即可；pipeline 内部完成读音频+推理
result = whisper_pipe(audio_filename)
# 字典里的 "text" 是完整转录字符串
transcription = result["text"]

# 打印分隔线与全文，并粗算词数（按空白 split）
print("=" * 60)
print("📝 TRANSCRIPTION RESULT:")
print("=" * 60)
print(transcription)
print(f"\n📊 Word count: {len(transcription.split())}")


---

## 🧠 第 2 部分：用 Llama 3.2 做摘要

把转录稿喂给 Llama，生成结构化 Markdown 摘要。

### 量化（Quantization）是什么？

Llama 3.2 3B 用 float16 大约要 ~6GB 显存。**4-bit 量化**把权重从 16-bit 压到 4-bit：

- 显存大约降到约 1/4（免费 Colab T4 更吃得消）
- 使用 NF4（Normal Float 4）时质量损失通常较小

`BitsAndBytesConfig` 就是控制这种压缩的开关。


In [ ]:
# ========== 配置 4-bit 量化（BitsAndBytes / NF4） ==========

# BitsAndBytesConfig：告诉 from_pretrained 如何压缩权重以省显存
quant_config = BitsAndBytesConfig(
    # 以 4-bit 格式加载权重（相对 float16 大约省约 4 倍）
    load_in_4bit=True,
    # 对量化常数再压缩一层（double quant），再省一点内存
    bnb_4bit_use_double_quant=True,
    # 前向计算用 bfloat16：精度与速度的折中
    bnb_4bit_compute_dtype=torch.bfloat16,
    # NF4（Normal Float 4）：更适合神经网络权重分布的 4-bit 格式
    bnb_4bit_quant_type="nf4",
)

print("✅ Quantization config ready!")


### 加载分词器与模型

- **Tokenizer**：人类文本 ↔ token id（每个模型有自己的词表）
- **Model**：真正的神经网络；一次预测下一个 token
- `device_map="auto"`：按显存自动把层放到 GPU/CPU


In [ ]:
# ========== 加载 Llama 分词器 + 4-bit 因果语言模型 ==========

# 按模型名从 Hub 拉分词器（含特殊 chat 标记）
tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL)
# Llama 常需把 pad_token 设成 eos_token，避免 padding 报错
tokenizer.pad_token = tokenizer.eos_token  # Required for Llama

# 加载模型：device_map=auto + 上面的 quant_config
print("⏳ Loading Llama 3.2 (4-bit quantized)... this takes about 1-2 minutes...")
model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL,
    device_map="auto",
    quantization_config=quant_config,
)

print("✅ Model loaded!")
# get_memory_footprint：粗看量化后占用多少 GB
print(f"📊 Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


### 构建提示（Prompt）

用两个角色的**聊天消息**：

- `system`：身份与输出格式规则
- `user`：具体任务（嵌入转录稿）

`apply_chat_template()` 会换成 Llama 期望的特殊标记格式（如 `<|begin_of_text|>` 等）。


In [ ]:
# ========== 组装 system/user prompt（英文原文保留，改译会改行为） ==========

# system：规定摘要结构（Overview / Topics / Quotes / Takeaways / Audience）
system_message = """
You are an expert podcast summarizer. Given a transcript of a podcast episode,
produce a well-structured summary in markdown (without code blocks) that includes:

1. **Episode Overview** — A 2-3 sentence high-level summary
2. **Key Topics Discussed** — Bullet points of the main subjects covered
3. **Notable Quotes** — Any memorable or impactful statements (in quotes)
4. **Key Takeaways** — The most important insights a listener should remember
5. **Who Should Listen** — What kind of audience would benefit from this episode

Keep it concise but comprehensive. Use a friendly, engaging tone.
"""

# user：嵌入上一阶段得到的 transcription
user_prompt = f"""
Below is a transcript from a podcast episode.
Please create a structured summary following the format specified.

Transcription:
{transcription}
"""

# OpenAI 风格 messages 列表，稍后交给 apply_chat_template
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt},
]

print("✅ Prompt ready!")
print(f"📊 System prompt: {len(system_message.split())} words")
print(f"📊 User prompt: {len(user_prompt.split())} words (includes transcription)")


### 生成摘要（流式）

`TextStreamer` 会在每个 token 生成时打印出来，所以你可以实时看到摘要「打字」出现。


In [ ]:
# ========== apply_chat_template → generate + TextStreamer 流式输出 ==========

# 把 messages 编成 token 张量，并放到 CUDA
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

# streamer：生成时边解码边打印到控制台
streamer = TextStreamer(tokenizer)

print("🧠 Generating podcast summary...\n")
print("=" * 60)

# max_new_tokens：摘要最长生成多少新 token；streamer 打开实时打印
outputs = model.generate(
    inputs,
    max_new_tokens=2000,  # Maximum length of the summary
    streamer=streamer,    # Stream tokens as they're generated
)

print("=" * 60)
print("\n✅ Summary generation complete!")


### 显示摘要（渲染 Markdown）

把完整生成序列解码后，尽量只留下 assistant 段落再 `display`。


In [ ]:
# ========== 解码完整序列，剥离特殊标记，渲染 Markdown ==========

# outputs[0] 是整段 token id（含提示+回答），decode 成字符串
full_response = tokenizer.decode(outputs[0])

# 尽量只取 assistant 段：Llama 聊天里常用 header 标出角色
if "assistant" in full_response:
    summary = full_response.split("assistant")[-1]
    # 清掉可能残留的特殊标记（字符串保持与原逻辑一致）
    summary = summary.replace("<|end_header_id|>", "")
    summary = summary.replace("<|eot_id|>", "")
    summary = summary.replace("<|begin_of_text|>", "")
    summary = summary.strip()
else:
    # 找不到标记时退回全文
    summary = full_response

# 标题字符串保持原样；拼上 summary 后渲染
display(Markdown("# 🎙️ Podcast Summary\n\n" + summary))


---

## 🔍 第 3 部分：对照「原文转录」与「AI 摘要」

并排查看：左边/上边是 Whisper 原文，右边/下边是结构化摘要，体会信息压缩。


In [ ]:
# ========== 并排展示：原始转录 vs AI 摘要 ==========

# 展示 Whisper 原文（Markdown 标题字符串保持原样）
display(Markdown("## 📝 Raw Transcription\n\n" + transcription))
print("\n" + "=" * 60 + "\n")
# 展示上一格得到的 summary
display(Markdown("## 🎯 AI-Generated Summary\n\n" + summary))


---

## 📊 加餐：播客统计

用词数粗算「压缩比」，并回显用过的模型与量化设置。


In [ ]:
# ========== 统计词数 / 压缩比，并用 Markdown 表格展示 ==========

# 按空白分词粗算（英文播客较合适；中文会偏粗）
transcript_words = len(transcription.split())
summary_words = len(summary.split())
# 压缩比例：摘要比原文短了多少百分比；避免除零
compression = (1 - summary_words / transcript_words) * 100 if transcript_words > 0 else 0

# f-string 拼出 Markdown 表；表内英文表头/模型 id 保持原样
stats = f"""
# # 📊 播客统计
# # 📊 Podcast Stats

| Metric | Value |
|--------|-------|
| Transcription words | {transcript_words:,} |
| Summary words | {summary_words:,} |
| Compression ratio | {compression:.1f}% reduction |
| Whisper model | {WHISPER_MODEL} |
| LLM model | {LLAMA_MODEL} |
| Quantization | 4-bit NF4 |
| Model memory | {model.get_memory_footprint() / 1e9:.2f} GB |
"""

display(Markdown(stats))


---

## 🎓 你学到了什么（HuggingFace 概念速查）

| 概念 | 做什么 | 本笔记用法 |
|------|--------|------------|
| **HuggingFace Hub** | 托管模型/数据集 | 下载 Whisper 与 Llama |
| **`pipeline()`** | 一行完成下载+推理 | Whisper 转录 |
| **`AutoTokenizer`** | 文本 ↔ token id | 把 prompt 编成 Llama 输入 |
| **`AutoModelForCausalLM`** | 加载文本生成模型 | Llama 3.2 |
| **`BitsAndBytesConfig`** | 4-bit 量化 | 塞进 T4 |
| **`TextStreamer`** | 实时打印生成 token | 流式看摘要 |
| **`apply_chat_template()`** | 转成模型专用聊天格式 | 组装 Llama 提示 |
| **`device_map="auto"`** | 自动切分到 GPU/CPU | 省心显存管理 |

### 数据流

```
Audio File (MP3)
    ↓ pipeline("automatic-speech-recognition")
Raw Text (Transcription)
    ↓ apply_chat_template() + tokenizer
Token IDs
    ↓ model.generate() with TextStreamer
Structured Summary (Markdown)
```
